# UFA Data Ingestion Pipeline

This notebook orchestrates the complete data pipeline:
1. Fetch metadata (teams, players, games) from UFA API
2. Clean and process game events
3. Insert everything into PostgreSQL database

**Prerequisites:**
- PostgreSQL running locally with `ufa_analytics` database
- Schema created (run `postgres_setup.ipynb` first)
- Functions defined in `data_cleaning_functions.ipynb`

## Setup and Configuration

In [9]:
# Import functions from other notebooks
import sys
sys.path.append('.')

# We'll import from the notebooks by running them
%run data_cleaning_functions.ipynb
%run postgres_setup.ipynb

Found 24 teams
Sample: {'team_id': 'aviators', 'team_name': 'LA Aviators', 'division': 'West', 'year': 2024}

Found 9 games in April 2024
Sample: {'game_id': '2024-04-27-MIN-PIT', 'home_team_id': 'thunderbirds', 'away_team_id': 'windchill', 'home_score': 11, 'away_score': 18, 'year': 2024, 'game_date': '2024-04-27', 'status': 'Final'}

Found 870 player-team records for 2024
Sample: {'player_id': 'mkiyoi', 'full_name': 'Michael Kiyoi', 'team_id': 'aviators', 'year': 2024, 'active': True, 'jersey_number': '1'}
Game 2024-04-27-MTL-NY: 750 events
First 5 events: [{'type': 1, 'line': ['cguay', 'swarrick', 'jbernat', 'rlalondel', 'tlalondel', 'jduquette', 'ctremblay'], 'time': 0, 'team': 'royal', 'year': 2024, 'gameID': '2024-04-27-MTL-NY'}, {'type': 2, 'line': ['lhaberfie', 'bjagt', 'skeegan', 'ochartock', 'srueschem', 'cweinberg', 'jwilliams'], 'time': 0, 'team': 'empire', 'year': 2024, 'gameID': '2024-04-27-MTL-NY'}, {'type': 7, 'puller': 'ctremblay', 'pullX': -16.47, 'pullY': 89.19, 'pul

In [ ]:
# Override connection to use Railway DB
# Set DATABASE_URL in your environment before running:
#   export DATABASE_URL="postgresql://..."
import os
import psycopg2

def get_connection():
    url = os.getenv('DATABASE_URL')
    if url:
        return psycopg2.connect(url, sslmode='require')
    return psycopg2.connect(**DB_CONFIG)  # fall back to local

print("✅ Connection configured (Railway if DATABASE_URL is set, local otherwise)")

In [11]:
from datetime import datetime
from tqdm import tqdm  # Progress bars
import time

### Configuration

Set your ingestion parameters here:

In [12]:
# Configuration
CONFIG = {
    'year': '2026',                     # Year to ingest
    'date_range': '2026-01:2026-12',    # Date range for games (or use year)
    'game_status': 'Final',             # Only get completed games
    'batch_size': 10,                   # Process games in batches
    'skip_existing': True,              # Reprocess all games with new coordinate averaging!
}

print("Pipeline Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

Pipeline Configuration:
  year: 2026
  date_range: 2026-01:2026-12
  game_status: Final
  batch_size: 10
  skip_existing: True


## Step 1: Fetch Teams and Players Metadata

In [13]:
print("Fetching teams data...")
teams = get_teams_data(years=CONFIG['year'])
print(f"✅ Fetched {len(teams)} teams")

# Display sample
if teams:
    print(f"\nSample team: {teams[0]}")

Fetching teams data...
✅ Fetched 22 teams

Sample team: {'team_id': 'apex', 'team_name': 'Colorado Apex', 'division': 'West', 'year': 2026}


In [14]:
print("Fetching players data...")
players = get_players_data(years=CONFIG['year'])
print(f"✅ Fetched {len(players)} player-team records")

# Display sample
if players:
    print(f"\nSample player: {players[0]}")

Fetching players data...
✅ Fetched 795 player-team records

Sample player: {'player_id': 'ctabor', 'full_name': 'Conor Tabor', 'team_id': 'apex', 'year': 2026, 'active': True, 'jersey_number': '0'}


### Insert Teams and Players into Database (Optional)

Note: The events insertion will work without pre-populating these tables,
but you can insert them here if you want to enforce foreign key constraints later.

In [15]:
# Insert teams and players into database
print("Inserting teams into database...")
insert_teams(teams)

print("\nInserting players into database...")
insert_players(players)

print("\n✅ Teams and players successfully inserted!")

Inserting teams into database...
✅ Inserted/updated 22 teams

Inserting players into database...
✅ Inserted/updated 795 players

✅ Teams and players successfully inserted!


## Step 2: Fetch Games List

In [16]:
print(f"Fetching games for {CONFIG['date_range']}...")
games = get_games(
    date=CONFIG['date_range'],
    statuses=CONFIG['game_status']
)

print(f"✅ Found {len(games)} games")

if games:
    print(f"\nDate range: {games[0]['game_date']} to {games[-1]['game_date']}")
    print(f"Sample game: {games[0]}")

Fetching games for 2026-01:2026-12...
✅ Found 67 games

Date range: 2026-05-09 to 2026-06-05
Sample game: {'game_id': '2026-05-09-MTL-TOR', 'home_team_id': 'rush', 'away_team_id': 'royal', 'home_score': 18, 'away_score': 17, 'year': 2026, 'game_date': '2026-05-09', 'status': 'Final'}


<cell_type>markdown</cell_type>## Step 3: Process and Insert Game Events

For each game:
1. Fetch game events from API
2. Clean and merge home/away events
3. Insert missing turnovers
4. **Normalize coordinates** (flip away team to home team's physical frame)
5. **Average turnover coordinates** (fixes team recording mismatches)
6. Insert into database

In [17]:
# Statistics tracking
stats = {
    'total_games': len(games),
    'processed': 0,
    'skipped': 0,
    'errors': 0,
    'total_events': 0,
    'synthetic_turnovers': 0
}

errors = []  # Track errors for review

In [18]:
print(f"\nProcessing {len(games)} games...\n")
print("=" * 80)

for i, game in enumerate(tqdm(games, desc="Processing games"), 1):
    game_id = game['game_id']
    
    try:
        # Check if game already exists (if skip_existing is enabled)
        if CONFIG['skip_existing']:
            conn = get_connection()
            cur = conn.cursor()
            cur.execute("SELECT COUNT(*) FROM games WHERE game_id = %s", (game_id,))
            exists = cur.fetchone()[0] > 0
            cur.close()
            conn.close()
            
            if exists:
                stats['skipped'] += 1
                continue
        
        # Fetch and clean events
        cleaned_data = clean_data([game_id])
        
        if not cleaned_data or game_id not in cleaned_data:
            print(f"\n⚠️ No data for game {game_id}")
            stats['errors'] += 1
            errors.append({'game_id': game_id, 'error': 'No event data returned'})
            continue
        
        events = cleaned_data[game_id]
        
        # Insert missing turnovers
        fixed_events = insert_missing_turnovers(events, seed=42)
        synthetic_count = len(fixed_events) - len(events)

        # Normalize coordinates (flip away team to home team's physical frame)
        normalized_events = normalize_coordinates(fixed_events, game['away_team_id'])

        # Average turnover coordinates (both teams now in same frame)
        averaged_events = average_turnover_coordinates(normalized_events)
        
        # Insert into database
        insert_game_data(
            game_id=game_id,
            events=averaged_events,
            home_team_id=game['home_team_id'],
            away_team_id=game['away_team_id'],
            home_score=game['home_score'],
            away_score=game['away_score'],
            year=game['year']
        )
        
        # Update stats
        stats['processed'] += 1
        stats['total_events'] += len(averaged_events)
        stats['synthetic_turnovers'] += synthetic_count
        
        # Progress update every 10 games
        if i % 10 == 0:
            print(f"\n✅ Processed {i}/{len(games)} games")
        
        # Brief pause to avoid overwhelming the API
        time.sleep(0.1)
        
    except Exception as e:
        print(f"\n❌ Error processing game {game_id}: {e}")
        stats['errors'] += 1
        errors.append({'game_id': game_id, 'error': str(e)})
        continue

print("\n" + "=" * 80)
print("Processing complete!")
print("=" * 80)


Processing 67 games...



Processing games:   0%|          | 0/67 [00:00<?, ?it/s]

Processing games: 100%|██████████| 67/67 [00:00<00:00, 342.01it/s]


Processing complete!


## Pipeline Summary

In [ ]:
print("\n" + "=" * 80)
print("PIPELINE SUMMARY")
print("=" * 80)
print(f"\nTotal games found: {stats['total_games']}")
print(f"Successfully processed: {stats['processed']}")
print(f"Skipped (already in DB): {stats['skipped']}")
print(f"Errors: {stats['errors']}")
print(f"\nTotal events inserted: {stats['total_events']:,}")
print(f"Synthetic turnovers added: {stats['synthetic_turnovers']}")

if stats['processed'] > 0:
    avg_events = stats['total_events'] / stats['processed']
    print(f"Average events per game: {avg_events:.1f}")

if errors:
    print(f"\n⚠️ {len(errors)} errors occurred:")
    for error in errors[:5]:  # Show first 5 errors
        print(f"  - {error['game_id']}: {error['error']}")
    if len(errors) > 5:
        print(f"  ... and {len(errors) - 5} more")

print("\n" + "=" * 80)


PIPELINE SUMMARY

Total games found: 67
Successfully processed: 0
Skipped (already in DB): 67
Errors: 0

Total events inserted: 0
Synthetic turnovers added: 0



## Verification Queries

In [ ]:
# Verify data was inserted correctly
conn = get_connection()
cur = conn.cursor()

# Count total games
cur.execute("SELECT COUNT(*) FROM games;")
total_games = cur.fetchone()[0]
print(f"Total games in database: {total_games}")

# Count total events
cur.execute("SELECT COUNT(*) FROM events;")
total_events = cur.fetchone()[0]
print(f"Total events in database: {total_events:,}")

# Count synthetic events
cur.execute("SELECT COUNT(*) FROM events WHERE synthetic = TRUE;")
synthetic_events = cur.fetchone()[0]
print(f"Synthetic events: {synthetic_events}")

# Count line player records
cur.execute("SELECT COUNT(*) FROM line_players;")
line_players_count = cur.fetchone()[0]
print(f"Line player records: {line_players_count:,}")

# Events by type
cur.execute("""
    SELECT event_type, COUNT(*) as count
    FROM events
    GROUP BY event_type
    ORDER BY event_type;
""")

print("\nEvents by type:")
for row in cur.fetchall():
    print(f"  Type {row[0]}: {row[1]:,} events")

cur.close()
conn.close()

Total games in database: 829
Total events in database: 610,931
Synthetic events: 176
Line player records: 550,594

Events by type:
  Type 1: 35,836 events
  Type 2: 35,820 events
  Type 3: 4,125 events
  Type 7: 33,046 events
  Type 8: 2,485 events
  Type 11: 15,931 events
  Type 16: 12,224 events
  Type 17: 12,227 events
  Type 18: 385,630 events
  Type 19: 32,831 events
  Type 20: 4,585 events
  Type 22: 26,092 events
  Type 23: 128 events
  Type 24: 443 events
  Type 25: 2,929 events
  Type 28: 1,646 events
  Type 29: 1,630 events
  Type 30: 1,636 events
  Type 31: 1,568 events
  Type 32: 92 events
  Type 33: 27 events


## Done! 🎉

Your UFA data has been successfully ingested into the PostgreSQL database.

### Next Steps:
- Query the database for analysis
- Build visualizations
- Create player/team statistics
- Analyze line combinations and synergies